# Trabajo final · Data Analyst · Luz del Valle

**Cooperativa eléctrica** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real de ARCA.

## La pregunta oficial

> **¿Cuánto nos cuesta la mora y a quién conviene contactar primero?**

**Decisión que habilita:** A qué usuarios y barrios dirigir la gestión de cobranza, y qué prevenir.

Antes de analizar, **verificá si los datos sostienen la premisa de la pregunta**. A veces no la sostienen, y decirlo es un hallazgo.

Las instrucciones completas (qué se entrega, rúbrica y defensa) están en el PDF **«Trabajo final · Data Analyst · Instrucciones»**. Este notebook es tu lugar de trabajo: vas del **SQL al análisis** sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 2 | SQL y PL/SQL |
| 3 | Estadística |
| 4 | Python: limpieza |
| 5 | Proceso de análisis |
| 6 | Superset |
| 7 | Analítica de negocio |
| 8 | Bitácora y entrega |

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib"}
print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")

## 1 · La base de Luz del Valle

La primera celda **genera la base** con código comprimido a propósito: no hace falta leerla, y leerla te quita la mitad del trabajo. La segunda la carga en SQLite y te deja tres herramientas: `sql(...)` para consultar, `esquema()` para ver las tablas y `anotar(...)` para dejar registrado cada problema de calidad que encuentres.

In [ ]:
import zlib, base64
_BLOB = (
    "eNq9V9tu2zYYvs9TEAI6UK2tSXbcJE7di27IZQYMLbDBMARWohJukiiQktskyFPtEfZi+35Skk9p0asZicTDf/z+AylVNdq0"
    "rO6q5oEJy+pmwhpR5xjir8nPclmw1HaZ0txMmJVGyQnbCqNE3UoL4nB5xvCzbOV3o0w3DzyMhG0fGsn1579k1obXrAKBiQxk"
    "64qXsuY2DNk71njudbXB/noUzLcTZkJWaMO2TNWOYHPNjGw7g9mZt6sSD51NpW0E2edYlgPNehuV+os0PIS9Udc0/ZC9YQEL"
    "JvTAcLtZm0hB4500lscTdh5uvPBcXKRl95jmsky3oiwlt1Lmq4v4ovfYwOC66T2KwCK6sk1NfecI4fEtCOZxHF+zz4KI10Xw"
    "4Ukt41n+HDjPFHkG/jvJkwlLknDjSI3SDqvsXqtMcqxM2C2QXq2j5HzCoiShx5weMT1meMSX4+hifLzdwAqrM2/ol3tpJMdA"
    "WVVzr2fC1sGHeE5wfIgvgg0AGmN0S+GJ5pcnS0kMsa1q9L7cYxqYEmS6kiZToiT5RlqVy9pNQwdgRznT5NGvohU3RlSSPwUq"
    "TzvbIQt0sCTAPhFgixcAuw03EOq9AOngTkB2YU4vmkFSIVJgQGqXBMaQmAg6vA0Kmd2LVJStwDaHNR9VJW0rqoYHszhZTOME"
    "f0GIXMFmq9MW+8iJVvDDxJlfxnjeQmZXq3YV/BqEYWRbUxADD179OX1VTV/lQfgcUpSt7FyQS31Xa1OJki+iBYK2WJAQ9nqH"
    "rEd6dQgnop9ADiyFlKdkyZJoDs4ZDc7JGhrEWDlfsugS7wW9sfEW7yu8L+iN9UtaR4yverok7gkSJ5QGTug8fnZBE1tp9ivZ"
    "hxtAocKlTcXWbY7AIA2uyKFrluka7aIQWYuEkllJBYEQ+v9r1hi59fn0KI22nFhAlp6sT9hnrUvaVTk2YmcVJUd1WE3zvkzp"
    "9/eXe5B60F8TaNRuXh+gP1QS6fXkYwCcyz8xXrH3q97J0M/fDXNkxwyRhxDw7kswuqtz7hdbYfbFvhjXhKAfSZCuiByicrUI"
    "IUBVzZFUOEFSoX1B2UeFaZHLB5XpEHxPthH/z7RRia+q6iq3RYnEpiwh7hGwRwiYzqMYkhF6aOFe7HuG7KJaiK6wSK2Fhguf"
    "rt/rKwPLGFLMnVwXNT6nxkuHhDbiJLsSGM0TcECH/Nrw6SORbtFLfP/YVWyBkp0tpk+Va7LTZDHULVqM/K0orGx5pev23q6S"
    "nbMnnWXpkjUSODMAM+9stGtLa4WEPdQDL4tSi5YCgm3kAdKfzwm8vUqYAuFZGJLpVASj9CK46eUMrQ4d5Ec1IqJeI6HxYrdx"
    "2VSLmqnCoQtyJksrGXcAvhkA9C0tFw92RdYfGI6yuCTLv6EgkHVKooMTHUEj7kQu0AlHsJuDGtjLKndgTVwroeLOTtKgMacR"
    "2x0/WbiON0vXFN6sWOKaR7kH8u+/PGHvRwGePh2BQMEDBgP6AYWwM7h2/PtPPUZjrMz9xpJEdDQ7/MLvOXDcUGf/q0OILB9v"
    "HGu0JNNK8hOJoXJtUMHhd72cuZMiig/d7Js3svSgmZvMUVCRHd8A/CmR6bKrarta798HnDWWXqgzeuVK2LQPRK5tsOlL61im"
    "P3MOZPZcJOVFDcpdix0C/oJAxaIqJetW7xaR3W5G3SfXTn9/sh3op7Uj/bQkKn2q3wkOhnvM4CTuTuDGHbeGkhE7wJ+tx6DR"
    "lZpWQzqx6YhtSpHJ1Y1AIYY9aHTHni7ifZjAkiGoaw+S60wWrRTX3beIqM/JFP61cuWutcgsRUGXqapz+XX10XQyHFgKVMQq"
    "eYktgguy9Tw8N7rxjNdMmwKG7OtNFkcC5mH/TeGoD3KCHAo+XdEv2O0O0XW7N3/QPR9b0W6HGtl6ttx8GwjQv+jpNSptPVw7"
    "Sf7u2wglOBx+hx8m1NTmPmz9t8lT0DtgcensrLsq17ar3NyXwNBhaMWbFPRJQys0fD47u5O1NO5WcfK1cvYfNjAtbg=="
)
exec(zlib.decompress(base64.b64decode(_BLOB)).decode("utf-8"))   # define generar(): código comprimido a propósito

In [ ]:
import sqlite3, matplotlib.pyplot as plt
pd.options.display.width = 200; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.3f}".format
t = generar(); tablas = {k: v for k, v in t.items() if not k.startswith("_")}
con = sqlite3.connect(":memory:")
for nombre, df in tablas.items(): df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False)

def sql(consulta, parametros=None):
    """Corre una consulta SQL sobre la base y devuelve un DataFrame."""
    return pd.read_sql_query(consulta, con, params=parametros)

def esquema():
    """Lista las tablas, sus columnas y sus filas."""
    for nombre, df in tablas.items(): print(f"{nombre}  ({len(df):,} filas)".replace(",", "."), "\n   ", ", ".join(df.columns))

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset"):
    """Guarda la base como archivo SQLite y un CSV por tabla, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); salida = sqlite3.connect(f"{carpeta}/base.db")
    for nombre, df in tablas.items(): df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, salida, index=False, if_exists="replace")
    salida.close(); print(f"Listo: {carpeta}/base.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta
import os
esquema()

Los datos tienen los problemas de un dato real. Encontrarlos, decidir qué hacer con cada uno y dejarlo escrito es parte de la evidencia.

## 2 · SQL y PL/SQL

Tres consultas. Guardalas en esta sección: son la evidencia del módulo.

1. La mora por barrio y su peso en lo facturado (JOIN de `facturas` con `usuarios`).
2. La mora según el salto de la factura contra el mes anterior (una función de ventana).
3. Un control de integridad: ¿hay facturas de usuarios que no existen?

SQLite no tiene PL/SQL. Si querés encapsular una lógica, escribila como una función de Python que ejecuta SQL.

In [ ]:
sql("""
-- Consulta 1
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 2
SELECT 1
""")

In [ ]:
sql("""
-- Consulta 3 (control de integridad)
SELECT 1
""")

## 3 · Estadística

Para la variable principal (`kwh`): **media, mediana, dispersión y distribución**. Decidí cuál resume mejor y escribí por qué.

In [ ]:
# Tu análisis estadístico

## 4 · Python: limpieza

Traé las tablas a pandas (`df = sql("SELECT * FROM tabla")`) y documentá la calidad: **faltantes, duplicados, valores imposibles y categorías inconsistentes**. Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)`.

In [ ]:
# Tu limpieza
# anotar("problema", "tabla.columna", 0, "qué decidimos y por qué")

In [ ]:
bitacora_calidad

## 5 · Proceso de análisis

**Verificá la premisa** de la pregunta oficial antes de seguir. Después armá al menos dos variables nuevas (variación de la factura contra el mes anterior, estación del año, reclamos previos, antigüedad del usuario).

In [ ]:
# Tu verificación de la premisa y tus variables nuevas

## 6 · Dashboard en Superset

Exportá la base y conectá Apache Superset al archivo. Armá **4 a 6 gráficos, uno por pregunta**, con un orden de lectura y las definiciones a la vista. Si no podés usar Superset, trabajá con gráficos de este notebook.

In [ ]:
exportar_superset()

In [ ]:
# Si no usás Superset, tus gráficos con matplotlib

## 7 · Analítica de negocio

**KPIs** con su denominador: tasa de mora, importe en mora, consumo por usuario, reclamos de facturación, concentración por barrio.

**Segmentación:** barrio, tipo de usuario, tarifa social y mes.

Un **insight** relevante, una **limitación** y una **recomendación** proporcional a la evidencia.

In [ ]:
# Tus KPIs y tu segmentación

## 8 · Bitácora y entrega

Completá esta bitácora editando la celda. Con ella, otra persona tiene que poder rehacer tus números.

**Equipo:** ____  **Integrantes y roles:** ____

**Pregunta y premisa:** ¿los datos sostienen la premisa? ____

| KPI | Fórmula | Universo | Período | Estados o faltantes |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |

**Hechos** (lo que muestran los datos): ____

**Inferencias** (lo que creemos que significa): ____

**Lo que contradice nuestra hipótesis:** ____

**Recomendación** (proporcional a la evidencia): ____

**Límites y siguiente análisis:** ____

### Antes de entregar

- [ ] Reinicié el kernel y corrí todo el notebook de punta a punta.
- [ ] Mi pregunta termina en una decisión y nombra quién la toma.
- [ ] Verifiqué la premisa de la pregunta.
- [ ] Cada KPI declara fórmula, universo, período y tratamiento de estados.
- [ ] Ningún porcentaje se muestra sin decir sobre cuántos casos.
- [ ] La limpieza está anotada: qué encontré, qué hice y cuántas filas afectó.
- [ ] Los hechos están separados de las inferencias y no prometo causalidad.
- [ ] Mostré al menos un resultado que no me conviene.
- [ ] Hay una limitación y un siguiente análisis.